# Transportation: the cheapest plan, and what it would take to change your mind

Two oil basins, three refineries. Each basin can supply so much, each refinery needs so much, and
every basin-to-refinery route has a cost per barrel. Find the cheapest way to move it all.

That has a one-line answer and it is not the interesting part. A linear program hands back three
more things for free, and they are what this notebook is about:

- for a route you are **not** using, how much cheaper would it have to get before you started?
- for a route you **are** using, how far can its price move before your plan changes shape?
- for a pipeline running at its limit, what is one more barrel of capacity worth per day?

Someone offering you a kickback to route through their pipeline is asking the first two. Someone
deciding whether to expand a pipeline is asking the third. The model answers all of them without
being re-solved — **but one of those answers depends on how you wrote a constraint, and the solver
will not tell you which.** That is the second half of this notebook.

## Licence setup

Three secrets named, none contained. Colab reads them from the key icon in the left sidebar; a
machine with a licence file needs nothing.

In [1]:
import gurobipy as gp

env = None
try:
    from google.colab import userdata
    try:
        params = {"WLSACCESSID": userdata.get("GRB_WLSACCESSID"),
                  "WLSSECRET":   userdata.get("GRB_WLSSECRET"),
                  "LICENSEID":   int(userdata.get("GRB_LICENSEID"))}
    except userdata.SecretNotFoundError:
        raise SystemExit("Add GRB_WLSACCESSID, GRB_WLSSECRET and GRB_LICENSEID as Colab Secrets "
                         "(key icon, left sidebar), then re-run this cell.")
    env = gp.Env(params=params)
    print("licence: Colab Secrets (WLS)")
except ImportError:
    env = gp.Env()
    print("licence: local gurobi.lic")

Set parameter Username


Set parameter LicenseID to value <removed>


Academic license - for non-commercial use only - expires 2026-12-04


licence: local gurobi.lic


## The instance is a table

Six routes with a cost each, five nodes with a quantity each. That is instance data — indexed by the
model's own sets, not something the narration names one entry at a time — so it lives in
`data/raw/` and both this notebook and the package read the same file.

Quantities are thousands of barrels per day; costs are dollars per barrel.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
from orteach import tolerance
from orteach.transportation import load_texas_crude

inst = load_texas_crude()

print(f"{'route':30} {'$/bbl':>7}")
for (o, d), c in inst.arcs.items():
    print(f"{o + ' -> ' + d:30} {c:7.2f}")
print()
print("supply (kbbl/d):", inst.supply)
print("demand (kbbl/d):", inst.demand)

route                            $/bbl
Permian -> Houston                3.00
Permian -> Corpus Christi         4.50
Permian -> Beaumont               3.50
Eagle Ford -> Houston             2.50
Eagle Ford -> Corpus Christi      1.50
Eagle Ford -> Beaumont            4.00

supply (kbbl/d): {'Permian': 100.0, 'Eagle Ford': 80.0}
demand (kbbl/d): {'Houston': 70.0, 'Corpus Christi': 60.0, 'Beaumont': 50.0}


Check the one structural fact that decides which form the model takes.

In [3]:
total_supply = sum(inst.supply.values())
total_demand = sum(inst.demand.values())
print(f"total supply {total_supply:.0f}   total demand {total_demand:.0f}   balanced: {inst.balanced}")

total supply 180   total demand 180   balanced: True


Supply exactly equals demand. Every barrel produced must ship and every refinery is exactly filled.
Hold on to that — it is going to matter more than it looks.

## Predict before building anything

Eagle Ford is the cheaper basin into Houston and Corpus Christi; Permian is cheaper into Beaumont.
But Eagle Ford only has 80 and Houston plus Corpus Christi want 130.

**Write down which basin you think serves Houston, and whether any refinery ends up drawing from
both basins.**

## The model, one piece at a time

One decision per route: how much to ship along it. Nothing else.

In [4]:
m = gp.Model(env=env)
tolerance.apply(m)          # solve as tightly as the assertion at the end claims
m.ModelSense = gp.GRB.MINIMIZE

ship = m.addVars(inst.arcs.keys(), lb=0.0, name="ship")
m.update()
print(f"{m.NumVars} variables, one per route")

6 variables, one per route


The objective is cost times quantity, summed over routes. This line reads the table rather than
restating it.

In [5]:
m.setObjective(gp.quicksum(c * ship[a] for a, c in inst.arcs.items()))
print(m.getObjective())

0.0


Now the rows. Here is the natural first instinct, and it is the one this notebook is going to take
apart: write supply as **at most** and demand as **at least**. It reads like plain English, it is
always feasible, and on a balanced instance the solver will make both hold with equality anyway —
so what could it matter?

In [6]:
supply_row = {}
for o, q in inst.supply.items():
    supply_row[o] = m.addConstr(
        gp.quicksum(ship[a] for a in inst.arcs if a[0] == o) <= q, name=f"supply[{o}]")
demand_row = {}
for d, q in inst.demand.items():
    demand_row[d] = m.addConstr(
        gp.quicksum(ship[a] for a in inst.arcs if a[1] == d) >= q, name=f"demand[{d}]")
m.update()
print(f"{m.NumConstrs} constraints: {len(supply_row)} supply (<=) + {len(demand_row)} demand (>=)")
assert m.NumConstrs == len(inst.supply) + len(inst.demand), "a row went missing"

5 constraints: 2 supply (<=) + 3 demand (>=)


**Predict before solving.** You wrote a routing down. Cheapest total, and does it use five routes or
fewer? A transportation problem with 2 + 3 nodes needs at most 2 + 3 − 1 = 4 routes in any basic
solution — so if your plan used five, one of them is unnecessary.

In [7]:
m.optimize()

plan = {a: ship[a].X for a in inst.arcs}
used = {a: f for a, f in plan.items() if f > 1e-9}

print(f"cheapest total: ${m.ObjVal:,.2f} per day\n")
for (o, d), f in sorted(used.items()):
    print(f"  {o + ' -> ' + d:30} {f:6.1f} kbbl/d")
print(f"\nroutes used: {len(used)}")

cheapest total: $465.00 per day

  Eagle Ford -> Corpus Christi     60.0 kbbl/d
  Eagle Ford -> Houston            20.0 kbbl/d
  Permian -> Beaumont              50.0 kbbl/d
  Permian -> Houston               50.0 kbbl/d

routes used: 4


Look at Houston against what you predicted. Both basins serve it, and the one doing most of the work
may not be the one you expected.

## The kickback question

Every route now carries a **reduced cost** — for a route not in the plan, how much its price would
have to fall before it enters — and a **ranging interval**: for a route in use, how far its price can
move before the plan changes shape.

Suppose someone with a stake in the Permian → Houston pipeline offers you a per-barrel kickback to
send more through it. **How big does the kickback have to be before it changes your plan?** The
ranging interval is supposed to answer exactly that.

In [8]:
print(f"{'route':30} {'cost':>6} {'reduced':>8} {'plan stable while cost in':>28}")
for a, c in inst.arcs.items():
    v = ship[a]
    lo = v.SAObjLow if v.SAObjLow > -1e20 else float("-inf")
    hi = v.SAObjUp  if v.SAObjUp  <  1e20 else float("inf")
    print(f"{a[0] + ' -> ' + a[1]:30} {c:6.2f} {v.RC:8.2f}   [{lo:6.2f}, {hi:6.2f}]")

reported_low = ship[("Permian", "Houston")].SAObjLow
print(f"\nreported: the plan changes once Permian -> Houston drops below ${reported_low:.2f}")
print(f"          i.e. a kickback of ${inst.arcs[('Permian', 'Houston')] - reported_low:.2f} per barrel")

route                            cost  reduced    plan stable while cost in
Permian -> Houston               3.00     0.00   [  2.50,   5.50]
Permian -> Corpus Christi        4.50     2.50   [  2.00,    inf]
Permian -> Beaumont              3.50     0.00   [  0.00,   4.50]
Eagle Ford -> Houston            2.50     0.00   [  0.00,   3.00]
Eagle Ford -> Corpus Christi     1.50     0.00   [ -0.50,   4.00]
Eagle Ford -> Beaumont           4.00     1.00   [  3.00,    inf]

reported: the plan changes once Permian -> Houston drops below $2.50
          i.e. a kickback of $0.50 per barrel


## Do not believe it. Test it.

The table says the plan survives until the Permian → Houston price falls to the reported bound. If
that is true, repricing to a nickel **below** the bound must change the plan. Re-solve and look.

In [9]:
def_cost = inst.arcs[("Permian", "Houston")]
probe = reported_low - 0.05

m_probe = gp.Model(env=env)
tolerance.apply(m_probe)
m_probe.ModelSense = gp.GRB.MINIMIZE
sp = m_probe.addVars(inst.arcs.keys(), lb=0.0, name="ship")
m_probe.setObjective(gp.quicksum((probe if a == ("Permian", "Houston") else c) * sp[a]
                                 for a, c in inst.arcs.items()))
for o, q in inst.supply.items():
    m_probe.addConstr(gp.quicksum(sp[a] for a in inst.arcs if a[0] == o) <= q)
for d, q in inst.demand.items():
    m_probe.addConstr(gp.quicksum(sp[a] for a in inst.arcs if a[1] == d) >= q)
m_probe.optimize()

probe_used = {a for a in inst.arcs if sp[a].X > 1e-9}
print(f"Permian -> Houston repriced to ${probe:.2f}, below the reported bound of ${reported_low:.2f}")
print(f"routes in the plan before: {sorted(used)}")
print(f"routes in the plan after : {sorted(probe_used)}")
print(f"plan changed: {probe_used != set(used)}")

Permian -> Houston repriced to $2.45, below the reported bound of $2.50
routes in the plan before: [('Eagle Ford', 'Corpus Christi'), ('Eagle Ford', 'Houston'), ('Permian', 'Beaumont'), ('Permian', 'Houston')]
routes in the plan after : [('Eagle Ford', 'Corpus Christi'), ('Eagle Ford', 'Houston'), ('Permian', 'Beaumont'), ('Permian', 'Houston')]
plan changed: False


The plan did **not** change. The reported bound was wrong, and nothing in the solver's output flagged
it. A kickback of that size would have been money for nothing.

## What went wrong: the basis is degenerate

Sensitivity ranging is a statement about the current **basis** — the particular set of variables the
simplex method is standing on — not about the solution. When the basis is degenerate, more than one
basis describes the same solution, and the ranging interval only covers the basis you happen to have.

And on a balanced instance, `<=` and `>=` rows guarantee degeneracy: every supply row binds, so every
supply **slack** is zero — but slacks are variables, and one of them is sitting in the basis at zero.
Look.

In [10]:
basic_slacks = [c.ConstrName for c in m.getConstrs() if c.CBasis == 0]
print("constraint rows whose slack is BASIC (in the basis):", basic_slacks)
for name in basic_slacks:
    row = m.getConstrByName(name)
    print(f"   {name}: slack = {row.Slack:.6f}   <- basic, and zero")

constraint rows whose slack is BASIC (in the basis): ['supply[Permian]']
   supply[Permian]: slack = 0.000000   <- basic, and zero


That is the degeneracy. The solver pivots on that zero slack at the reported bound — a basis change
with no solution change — and honestly reports the bound for the basis it had.

**The fix is the classical form.** On a balanced instance, write the rows as equalities. There are
then no supply or demand slacks to sit in the basis, and the ranging interval describes the solution.

In [11]:
m_eq = gp.Model(env=env)
tolerance.apply(m_eq)
m_eq.ModelSense = gp.GRB.MINIMIZE
ship_eq = m_eq.addVars(inst.arcs.keys(), lb=0.0, name="ship")
m_eq.setObjective(gp.quicksum(c * ship_eq[a] for a, c in inst.arcs.items()))
for o, q in inst.supply.items():
    m_eq.addConstr(gp.quicksum(ship_eq[a] for a in inst.arcs if a[0] == o) == q, name=f"supply[{o}]")
for d, q in inst.demand.items():
    m_eq.addConstr(gp.quicksum(ship_eq[a] for a in inst.arcs if a[1] == d) == q, name=f"demand[{d}]")
m_eq.optimize()

plan_eq = {a: ship_eq[a].X for a in inst.arcs}
true_low = ship_eq[("Permian", "Houston")].SAObjLow
print(f"objective {m_eq.ObjVal:,.2f}   same plan as before: {plan_eq == plan}")
print(f"ranging low for Permian -> Houston, equality form: ${true_low:.2f}")
print(f"kickback required: ${def_cost - true_low:.2f} per barrel   (the <= form said ${def_cost - reported_low:.2f})")

objective 465.00   same plan as before: True
ranging low for Permian -> Houston, equality form: $2.00
kickback required: $1.00 per barrel   (the <= form said $0.50)


Same objective, same plan, a different answer to the question that mattered. Test this one the same
way — a nickel below the new bound, and a nickel above it.

In [12]:
def_used = set(used)
outcomes = {}
for label, price in (("just above", true_low + 0.05), ("just below", true_low - 0.05)):
    mm = gp.Model(env=env)
    tolerance.apply(mm)
    mm.ModelSense = gp.GRB.MINIMIZE
    ss = mm.addVars(inst.arcs.keys(), lb=0.0)
    mm.setObjective(gp.quicksum((price if a == ("Permian", "Houston") else c) * ss[a]
                                for a, c in inst.arcs.items()))
    for o, q in inst.supply.items():
        mm.addConstr(gp.quicksum(ss[a] for a in inst.arcs if a[0] == o) == q)
    for d, q in inst.demand.items():
        mm.addConstr(gp.quicksum(ss[a] for a in inst.arcs if a[1] == d) == q)
    mm.optimize()
    outcomes[label] = {a for a in inst.arcs if ss[a].X > 1e-9}
    print(f"{label} the bound (${price:.2f}): plan changed = {outcomes[label] != def_used}")

assert outcomes["just above"] == def_used, "plan changed inside its own ranging interval"
assert outcomes["just below"] != def_used, "plan did not change past its ranging bound"
print("\nthe equality-form bound is the real one: stable above it, different below it")

just above the bound ($2.05): plan changed = False
just below the bound ($1.95): plan changed = True

the equality-form bound is the real one: stable above it, different below it


Now the answer holds up. A kickback smaller than that does nothing; one that large re-routes
everything at once, with Permian flooding Houston and Eagle Ford's barrels pushed to Beaumont.

Two things to carry away. **The plan is stable across a range, and the range has an edge.** And
**sensitivity output is only as trustworthy as the basis it came from** — which is why the package
version below uses equality rows on any balanced instance and inequalities only where the instance
actually needs the slack.

## A pipeline at its limit

Add the constraint the physical world imposes: the Eagle Ford → Corpus Christi pipeline can carry at
most 50 kbbl/d, and the cheapest plan wanted more.

**Predict:** by how much does the daily cost rise, and which route absorbs the displaced barrels?

In [13]:
pipeline = ("Eagle Ford", "Corpus Christi")
cap = 50.0

m2 = gp.Model(env=env)
tolerance.apply(m2)
m2.ModelSense = gp.GRB.MINIMIZE
ship2 = m2.addVars(inst.arcs.keys(), lb=0.0, name="ship")
m2.setObjective(gp.quicksum(c * ship2[a] for a, c in inst.arcs.items()))
for o, q in inst.supply.items():
    m2.addConstr(gp.quicksum(ship2[a] for a in inst.arcs if a[0] == o) == q, name=f"supply[{o}]")
for d, q in inst.demand.items():
    m2.addConstr(gp.quicksum(ship2[a] for a in inst.arcs if a[1] == d) == q, name=f"demand[{d}]")
pipe_row = m2.addConstr(ship2[pipeline] <= cap, name="pipeline_cap")
m2.optimize()

plan2 = {a: ship2[a].X for a in inst.arcs}
print(f"with the pipeline cap: ${m2.ObjVal:,.2f} per day   (+${m2.ObjVal - m_eq.ObjVal:,.2f})\n")
for (o, d), f in sorted(plan2.items()):
    if f > 1e-9:
        print(f"  {o + ' -> ' + d:30} {f:6.1f}")
print(f"\npipeline flow {plan2[pipeline]:.1f} against a cap of {cap:.0f}")

with the pipeline cap: $490.00 per day   (+$25.00)

  Eagle Ford -> Corpus Christi     50.0
  Eagle Ford -> Houston            30.0
  Permian -> Beaumont              50.0
  Permian -> Corpus Christi        10.0
  Permian -> Houston               40.0

pipeline flow 50.0 against a cap of 50


The cap binds — the pipeline is full — so it carries a **shadow price**: the change in daily cost per
unit of capacity. Negative in a minimisation, because more capacity can only help. Someone deciding
whether to expand this pipeline wants exactly this number.

In [14]:
pi = pipe_row.Pi
print(f"shadow price on the pipeline cap : {pi:+.3f} $/bbl per day")
print(f"one more kbbl/d of capacity is worth : ${-pi * 1000:,.0f} per day")
assert pi <= tolerance.FEASIBILITY_ATOL, "a capacity's shadow price cannot be positive in a minimisation"

shadow price on the pipeline cap : -2.500 $/bbl per day
one more kbbl/d of capacity is worth : $2,500 per day


## A second instance, where the units are the trap

Three distribution centres, five dealers, distances in miles. Its data are in **units** while trucks
carry **18** at a time and cost **$25 per mile** — so a factor of 18 and a factor of 25 have to
appear somewhere, and where they appear is a choice.

The version this came from divided by 18 inside the constraint expressions, as a bare number. Here
the truckload is a **named knob** the loader applies once, and the tariff is a **named knob** the
solve applies once. Neither is typed into a constraint.

In [15]:
from orteach.transportation import load_dc_dealers

truckload = 18          # units per truck - the number that used to be an unexplained /18
cost_per_truck_mile = 25

dc = load_dc_dealers(truckload=truckload)
print("supply in trucks:", {k: round(v, 2) for k, v in dc.supply.items()})
print("demand in trucks:", {k: round(v, 2) for k, v in dc.demand.items()})
print(f"balanced: {dc.balanced}   -> equality rows")

supply in trucks: {'1': 22.22, '2': 11.11, '3': 8.33}
demand in trucks: {'1': 5.56, '2': 11.11, '3': 8.33, '4': 8.89, '5': 7.78}
balanced: True   -> equality rows


**Predict:** the answer comes back in truckloads. Will every number be a whole truck?

In [16]:
m3 = gp.Model(env=env)
tolerance.apply(m3)
m3.ModelSense = gp.GRB.MINIMIZE
ship3 = m3.addVars(dc.arcs.keys(), lb=0.0, name="trucks")
m3.setObjective(gp.quicksum(cost_per_truck_mile * miles * ship3[a] for a, miles in dc.arcs.items()))
for o, q in dc.supply.items():
    m3.addConstr(gp.quicksum(ship3[a] for a in dc.arcs if a[0] == o) == q, name=f"supply[{o}]")
for d, q in dc.demand.items():
    m3.addConstr(gp.quicksum(ship3[a] for a in dc.arcs if a[1] == d) == q, name=f"demand[{d}]")
m3.optimize()

plan3 = {a: ship3[a].X for a in dc.arcs}
print(f"cheapest: ${m3.ObjVal:,.2f}\n")
fractional = 0
for (o, d), f in sorted(plan3.items()):
    if f > 1e-9:
        whole = abs(f - round(f)) < 1e-6
        fractional += 0 if whole else 1
        print(f"  DC {o} -> dealer {d}: {f:7.3f} trucks{'' if whole else '   <- not a whole truck'}")
print(f"\nroutes with a fractional truck: {fractional}")

cheapest: $87,916.67

  DC 1 -> dealer 2:   5.556 trucks   <- not a whole truck
  DC 1 -> dealer 4:   8.889 trucks   <- not a whole truck
  DC 1 -> dealer 5:   7.778 trucks   <- not a whole truck
  DC 2 -> dealer 2:   2.778 trucks   <- not a whole truck
  DC 2 -> dealer 3:   8.333 trucks   <- not a whole truck
  DC 3 -> dealer 1:   5.556 trucks   <- not a whole truck
  DC 3 -> dealer 2:   2.778 trucks   <- not a whole truck

routes with a fractional truck: 7


An LP does not know that trucks come whole. Where the plan says 5.56 trucks it means 100 units, and
somebody has to decide whether that is six trucks with one part-empty or five trucks and 10 units
left behind. Making the model decide that is an integer program — the next folder — and it is where
transportation stops being a linear problem.

The assignment notebook beside this one explains why the crude instance came out whole and this one
did not, though both are transportation problems.

---

# Now the streamlined version

Four models built by hand from the same blocks, so they belong in one function now. The package
solver returns the plan **and** the sensitivity information, uses **equality rows on balanced
instances** for the reason you just tested, and takes the instance as an argument rather than reading
a file.

In [17]:
from orteach import transportation as tp
from orteach.tolerance import AGREEMENT_RTOL, rel_diff

pkg_plain  = tp.solve(inst, env=env)
pkg_capped = tp.solve(tp.with_arc_cap(inst, pipeline, cap), env=env)
pkg_dc     = tp.solve(dc, cost_multiplier=cost_per_truck_mile, env=env)

print(f"{'':18} {'objective':>12}")
print(f"{'Texas crude':18} {pkg_plain.objective:12.2f}   ranging low P->H {pkg_plain.obj_low[('Permian', 'Houston')]:.2f}")
print(f"{'  with pipe cap':18} {pkg_capped.objective:12.2f}   shadow price {pkg_capped.cap_price[pipeline]:+.3f}")
print(f"{'DC to dealers':18} {pkg_dc.objective:12.2f}")

                      objective
Texas crude              465.00   ranging low P->H 2.00
  with pipe cap          490.00   shadow price -2.500
DC to dealers          87916.67


## The agreement assertion

The three equality-form hand-built models against the package, number by number — objectives, every
flow, the shadow price, and the ranging bound the kickback argument rested on. Both sides solved at
the same tightened tolerances, so agreement to `AGREEMENT_RTOL` is a claim the computation supports.
The inequality-form model is deliberately **not** compared: its ranging bound is the wrong number,
and the check exists to catch wrong numbers.

In [18]:
checks = [("Texas objective",  m_eq.ObjVal, pkg_plain.objective),
          ("capped objective", m2.ObjVal,   pkg_capped.objective),
          ("DC objective",     m3.ObjVal,   pkg_dc.objective),
          ("pipeline shadow price", pi, pkg_capped.cap_price[pipeline]),
          ("Permian->Houston ranging low", true_low, pkg_plain.obj_low[("Permian", "Houston")])]
for a in inst.arcs:
    checks.append((f"flow {a[0]}->{a[1]}", plan_eq[a], pkg_plain.flow[a]))
    checks.append((f"capped flow {a[0]}->{a[1]}", plan2[a], pkg_capped.flow[a]))
for a in dc.arcs:
    checks.append((f"DC flow {a}", plan3[a], pkg_dc.flow[a]))

worst = max(rel_diff(h, p) for _, h, p in checks)
print(f"{len(checks)} comparisons")
for name, hand, packaged in checks[:5]:
    print(f"  {name:30} hand {hand:12.4f}   package {packaged:12.4f}   rel {rel_diff(hand, packaged):.2e}")
print("  ...")
assert worst < AGREEMENT_RTOL, f"notebook and package disagree by {worst:.2e}"
print(f"\nnotebook and package agree to {worst:.1e}")

32 comparisons
  Texas objective                hand     465.0000   package     465.0000   rel 0.00e+00
  capped objective               hand     490.0000   package     490.0000   rel 0.00e+00
  DC objective                   hand   87916.6667   package   87916.6667   rel 0.00e+00
  pipeline shadow price          hand      -2.5000   package      -2.5000   rel 0.00e+00
  Permian->Houston ranging low   hand       2.0000   package       2.0000   rel 0.00e+00
  ...

notebook and package agree to 0.0e+00


---

## Where to take this next

- Raise the pipeline cap from 50 to 60 and re-solve. Does the shadow price stay the same, and what
  does that tell you about how far the "worth per kbbl/d" figure can be trusted?
- Give Eagle Ford 10 more kbbl/d of supply. The instance is now unbalanced, the package switches to
  inequality rows — and now the slack is real. Which rows go slack, and what happens to their shadow
  prices?
- Go back to the `<=` model and ask Gurobi for `SAObjUp` on Permian → Houston instead of `SAObjLow`.
  Is that bound also wrong, and can you work out from the basis why or why not?